In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  STEP 1 · VISIBLE DEPENDENCY INJECTION & LINKER FORGE                    ║
# ╚══════════════════════════════════════════════════════════════════════════╝

# 1. Nuke conflicting bindings
!pip uninstall -y transformers torchvision torchaudio protobuf vllm

# 2. SOTA FIX: Threading the dependency needle
# vLLM 0.16.0+ requires protobuf >= 5.29.6. We cap it at < 6.0.0 to prevent 
# the v7.x MessageFactory crashes from destroying the Google GenAI libraries.
!pip install --no-cache-dir "vllm>=0.16.0" fastembed-gpu "sentence-transformers>=2.7.0" "nest-asyncio>=1.6.0" "fastapi>=0.111.0" "uvicorn[standard]>=0.30.0" "httpx>=0.27.0" outlines==0.0.34 tenacity pydantic "transformers>=4.57.1" accelerate "protobuf>=5.29.6,<6.0.0"

# 3. AGGRESSIVE LINKER FORGE: Fix the "cannot find -lcuda" error
import subprocess, os
print("🛠️ Forging CUDA Linker symlinks...")
driver_path = "/usr/lib/x86_64-linux-gnu/libcuda.so.1"
if not os.path.exists(driver_path):
    driver_path = "/usr/local/nvidia/lib64/libcuda.so.1"

if os.path.exists(driver_path):
    subprocess.run(f"ln -sf {driver_path} /usr/lib/x86_64-linux-gnu/libcuda.so", shell=True)
    subprocess.run(f"ln -sf {driver_path} /usr/local/cuda/lib64/libcuda.so", shell=True)
    print(f"✅ Linker forged using driver at: {driver_path}")
else:
    print("⚠️ WARNING: libcuda.so.1 not found. Check GPU settings.")

print("\n✅ INSTALL COMPLETE. RESTART KERNEL NOW.")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  STEP 2 · MASTER BOOTLOADER (Qwen3-VL-8B-Instruct-FP8 + Tunnel)          ║
# ║  SOTA FIX: Persistent Cache, Self-Healing & Console Streaming Restored   ║
# ╚══════════════════════════════════════════════════════════════════════════╝
import os, subprocess, torch, torch.multiprocessing as mp, logging, uuid, nest_asyncio, uvicorn, re, time
import base64, io, asyncio, shutil
from PIL import Image
from fastapi import FastAPI, Request, Depends, HTTPException, Security
from fastapi.responses import StreamingResponse
from fastapi.security.api_key import APIKeyHeader

# ── 1. ENVIRONMENT LOCKDOWN ────────────────────────────────────────────────
os.environ["VLLM_USE_V1"] = "0"
os.environ["VLLM_ATTENTION_BACKEND"] = "XFORMERS" 
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"
os.environ["RAY_IGNORE_UNHANDLED_ERRORS"] = "1"

# 🚨 CACHE REDIRECTION: Protect models from volatile /tmp cleanup
PERSISTENT_CACHE = "/kaggle/working/fastembed_cache"
os.makedirs(PERSISTENT_CACHE, exist_ok=True)

mp.set_sharing_strategy('file_system')
nest_asyncio.apply()

# ── 2. CONFIGURATION ──────────────────────────────────────────────────────
TUNNEL_API_KEY = "omni_colab_secret_123" 
TARGET_MODEL = "Qwen/Qwen3-VL-8B-Instruct-FP8" 
SERVER_PORT = 8000
CLOUDFLARED_LOG = "/kaggle/working/cloudflared.log"

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(message)s")
logger = logging.getLogger("ATLAS.Master")
app = FastAPI(title="ATLAS Sovereign Edge Node (Qwen3-VL)")
api_key_header = APIKeyHeader(name="X-API-Key", auto_error=True)

is_processing = False

# ── 3. ENGINE INITIALIZATION ──────────────────────────────────────────────
from vllm import AsyncEngineArgs, AsyncLLMEngine, SamplingParams
from fastembed import LateInteractionTextEmbedding
from sentence_transformers import CrossEncoder

logger.info(f"🚀 Booting Stage 1: {TARGET_MODEL} (Isolated to GPU 0)...")

llm_engine = AsyncLLMEngine.from_engine_args(AsyncEngineArgs(
    model=TARGET_MODEL, 
    tensor_parallel_size=1,
    gpu_memory_utilization=0.96,
    max_model_len=16384,
    kv_cache_dtype="fp8",
    enable_prefix_caching=True,
    enable_chunked_prefill=True, 
    max_num_batched_tokens=4096,
    enforce_eager=True, 
    disable_custom_all_reduce=True, 
    trust_remote_code=True,
    limit_mm_per_prompt={"image": 1} 
))

from transformers import AutoProcessor
logger.info("🛠️ Loading Qwen-VL Native Processor...")
processor = AutoProcessor.from_pretrained(TARGET_MODEL, trust_remote_code=True)

logger.info("🚀 Booting Stage 2: Embedder & Reranker (Isolated to GPU 1)...")

# 🚨 SELF-HEALING BLOCK: Wipe corrupted /tmp residues and use persistent storage
try:
    shutil.rmtree("/tmp/fastembed_cache", ignore_errors=True)
    embedder = LateInteractionTextEmbedding(
        "jinaai/jina-colbert-v2", 
        cache_dir=PERSISTENT_CACHE,
        providers=[("CUDAExecutionProvider", {"device_id": 1})]
    )
except Exception as e:
    logger.warning(f"⚠️ Cache corruption detected: {e}. Performing hard reset...")
    shutil.rmtree(PERSISTENT_CACHE, ignore_errors=True)
    os.makedirs(PERSISTENT_CACHE, exist_ok=True)
    embedder = LateInteractionTextEmbedding(
        "jinaai/jina-colbert-v2", 
        cache_dir=PERSISTENT_CACHE,
        providers=[("CUDAExecutionProvider", {"device_id": 1})]
    )

reranker = CrossEncoder(
    "BAAI/bge-reranker-v2-m3", 
    device="cuda:1",
    model_kwargs={"torch_dtype": torch.float16}
)

# ── 4. ENDPOINTS ───────────────────────────────────────────────────────────
def verify_key(api_key: str = Security(api_key_header)):
    if api_key != TUNNEL_API_KEY: raise HTTPException(403, "Invalid API Key")

@app.post("/generate", dependencies=[Depends(verify_key)])
async def gen(r: Request):
    global is_processing
    is_processing = True
    data = await r.json()
    sp = SamplingParams(temperature=data.get("temperature", 0.0), max_tokens=data.get("max_tokens", 4096), top_p=0.95)
    prompt_text = data.get("prompt", "")
    system_instruction = data.get("system_instruction", "")
    image_b64 = data.get("image_base64")
    
    sys_part = f"<|im_start|>system\n{system_instruction}<|im_end|>\n" if system_instruction else ""
    if image_b64:
        user_part = f"<|im_start|>user\n<|vision_start|><|image_pad|><|vision_end|>\n{prompt_text}<|im_end|>\n"
    else:
        user_part = f"<|im_start|>user\n{prompt_text}<|im_end|>\n"
    
    vllm_inputs = {"prompt": sys_part + user_part + "<|im_start|>assistant\n"}
    
    if image_b64:
        try:
            if image_b64.startswith("data:image"): image_b64 = image_b64.split(",", 1)[1]
            img = Image.open(io.BytesIO(base64.b64decode(image_b64))).convert("RGB")
            vllm_inputs["multi_modal_data"] = {"image": img}
        except Exception as e: logger.error(f"Image decode error: {e}")

    res = llm_engine.generate(vllm_inputs, sp, f"at-{uuid.uuid4().hex[:8]}")
    
    async def stream():
        global is_processing
        last = ""
        logger.info("📡 Request Received: Generating Output...")
        try:
            async for out in res:
                text = out.outputs[0].text
                new_tokens = text[len(last):]
                if new_tokens:
                    # 🟢 RESTORED: Real-time console printing
                    print(new_tokens, end="", flush=True) 
                    yield new_tokens
                last = text
                await asyncio.sleep(0.01)
            print("\n✅ Generation Complete.")
        finally: is_processing = False
            
    return StreamingResponse(stream(), media_type="text/event-stream", headers={"X-Accel-Buffering": "no", "Connection": "keep-alive"})

@app.post("/embed", dependencies=[Depends(verify_key)])
async def embed(r: Request):
    data = await r.json()
    return {"vectors":[e.tolist() for e in list(embedder.embed(data.get("texts", [])))]}

@app.post("/rerank", dependencies=[Depends(verify_key)])
async def rerank(r: Request):
    data = await r.json()
    scores = reranker.predict([[data.get("query"), c] for c in data.get("chunks",[])]).tolist()
    return {"scores": scores, "ranked_indices": sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)}

@app.get("/health", dependencies=[Depends(verify_key)])
async def health(): return {"status": "online", "model": TARGET_MODEL}

async def tunnel_heartbeat():
    while True:
        if not is_processing: print(" ", end="", flush=True)
        await asyncio.sleep(30)

# ── 5. CONNECTIVITY & LAUNCH ───────────────────────────────────────────────
logger.info("📡 Initializing Cloudflare Tunnel...")
subprocess.Popen(f"wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared", shell=True).wait()
if os.path.exists(CLOUDFLARED_LOG): os.remove(CLOUDFLARED_LOG)
subprocess.Popen(f"./cloudflared tunnel --url http://127.0.0.1:{SERVER_PORT} > {CLOUDFLARED_LOG} 2>&1", shell=True)

for _ in range(20):
    if os.path.exists(CLOUDFLARED_LOG):
        m = re.search(r"https://[a-z0-9\-]+\.trycloudflare\.com", open(CLOUDFLARED_LOG, errors="replace").read())
        if m: 
            print(f"\n🔱 SOVEREIGN NODE ACCESS URL: {m.group(0)}\n")
            break
    time.sleep(2)

config = uvicorn.Config(app=app, host="0.0.0.0", port=SERVER_PORT, log_level="warning", loop="asyncio")
server = uvicorn.Server(config)
loop = asyncio.get_event_loop()
loop.create_task(tunnel_heartbeat())
await server.serve()

2026-05-01 14:35:54,313 | NumExpr defaulting to 4 threads.
2026-05-01 14:35:59,066 | TensorFlow version 2.19.0 available.
2026-05-01 14:35:59,069 | JAX version 0.7.2 available.
2026-05-01 14:35:59,417 | 🚀 Booting Stage 1: Qwen/Qwen3-VL-8B-Instruct-FP8 (Isolated to GPU 0)...


WARNING 05-01 14:35:59 [envs.py:1818] Unknown vLLM environment variable detected: VLLM_USE_V1
WARNING 05-01 14:35:59 [envs.py:1818] Unknown vLLM environment variable detected: VLLM_ATTENTION_BACKEND


2026-05-01 14:35:59,562 | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


INFO 05-01 14:35:59 [model.py:555] Resolved architecture: Qwen3VLForConditionalGeneration
WARNING 05-01 14:35:59 [model.py:1965] Your device 'Tesla T4' (with compute capability 7.5) doesn't support torch.bfloat16. Falling back to torch.float16 for compatibility.
WARNING 05-01 14:35:59 [model.py:2018] Casting torch.bfloat16 to torch.float16.
INFO 05-01 14:35:59 [model.py:1680] Using max model len 16384
INFO 05-01 14:36:00 [nixl_utils.py:20] Setting UCX_RCACHE_MAX_UNRELEASED to '1024' to avoid a rare memory leak in UCX when using NIXL.
WARNING 05-01 14:36:00 [nixl_utils.py:34] NIXL is not available
WARNING 05-01 14:36:00 [nixl_utils.py:44] NIXL agent config is not available
INFO 05-01 14:36:01 [cache.py:261] Using fp8 data type to store kv cache. It reduces the GPU memory footprint and boosts the performance. Meanwhile, it may cause accuracy drop without a proper scaling factor
INFO 05-01 14:36:01 [scheduler.py:239] Chunked prefill is enabled with max_num_batched_tokens=4096.
INFO 05-01 

[transformers] `Qwen2VLImageProcessorFast` is deprecated. The `Fast` suffix for image processors has been removed; use `Qwen2VLImageProcessor` instead.
[transformers] The `use_fast` parameter is deprecated and will be removed in a future version. Use `backend="torchvision"` instead of `use_fast=True`, or `backend="pil"` instead of `use_fast=False`.


...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]